# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a dataset described with a Croissant schema using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a [Croissant schema JSON-LD](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json). It comprises structured tabulations of clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.

*Key variables and meta-information (from Croissant package):*
- Record sets, fields, and columns can be referenced by their `@id` for clarity.
- Use variables for dataset paths and field references, as modeled in this notebook.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema describes three main tabular record sets:
- Table 1: Clinical features (@id: `table1-clinical-features`)
- Table 2: Hormone levels & imaging (@id: `table2-hormone-imaging`)
- Table 3: Genetic variants (@id: `table3-genetic-variants`)

Below, we inspect the schema's record sets, listing their `@id` and fields, by accessing metadata. All exploration is done referencing entities by their `@id`.

In [ ]:
# List record sets, fields, and columns by @id

record_sets = dataset.record_sets
print("Available record sets (by @id):")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")
    print("  Fields:")
    for field in rs.get('field', []):
        print(f"     - Field @id: {field['@id']} [name: {field.get('name', field['@id'])}] (dataType: {field.get('dataType', '-')})")
    print()

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.

Each record set is referenced explicitly by its `@id`. All field operations use their `@id` as DataFrame column keys.

In [ ]:
# Extract data from all record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nColumns for RecordSet {rs_id}:")
    print(df.columns.tolist())
    print("Sample records:")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data filtering and transformation steps.

We'll demonstrate:
1. Filtering patients (from Table 1) with age above a threshold.
2. Normalizing numeric fields.
3. Grouping data (if categorical field available).

> All fields referenced by their `@id`.

In [ ]:
# EDA on Table 1: Clinical features
table1_id = record_set_ids[0]  # typically Table 1
df1 = dataframes[table1_id]

# Example numeric field: age at diagnosis (assumed @id 'age_at_diagnosis')
age_field_id = 'age_at_diagnosis'

# Set filtering threshold
age_threshold = 20
filtered_df = df1[df1[age_field_id] > age_threshold]

print(f"Filtered records in {table1_id} with {age_field_id} > {age_threshold}:")
print(filtered_df.head())

# Normalizing age
filtered_df[f"{age_field_id}_normalized"] = (
    filtered_df[age_field_id] - filtered_df[age_field_id].mean()
) / filtered_df[age_field_id].std()
print(f"Normalized {age_field_id} for filtered records:")
print(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

# If a categorical field exists (e.g., 'hirsutism'), group by it
group_field_id = 'hirsutism'  # example field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[age_field_id].mean().reset_index()
    print(f"Grouped filtered records by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize selected numeric and categorical relationships using matplotlib.

Below: Distribution of age at diagnosis and boxplot for age grouped by hirsutism status.

In [ ]:
# Plot age at diagnosis distribution
plt.figure(figsize=(6, 4))
df1[age_field_id].plot(kind='hist', bins=10, color='skyblue', edgecolor='black')
plt.title('Age at Diagnosis Distribution')
plt.xlabel('Age (years)')
plt.ylabel('Count')
plt.grid(True)
plt.show()

# Boxplot: Age at diagnosis by hirsutism (if exists)
if group_field_id in df1.columns:
    plt.figure(figsize=(6, 4))
    df1.boxplot(column=age_field_id, by=group_field_id)
    plt.title(f'Age at Diagnosis by {group_field_id.capitalize()}')
    plt.suptitle('')
    plt.xlabel(group_field_id)
    plt.ylabel('Age')
    plt.show()

## 6. Conclusion
This notebook demonstrates how to load, process, and visualize a Croissant-described dataset using the `mlcroissant` library. Key findings:
- Data can be referenced and extracted using `@id` fields for record sets and fields.
- Simple EDA and normalization are possible even on small datasets.
- Visualization provides insights into attribute distributions and relations like age and hirsutism status.

Further analysis could include examining hormone data (Table 2) or genetic variants (Table 3) by referencing their respective `@id`.

> For more, see: https://mlcommons.org/croissant/